In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df=pd.read_csv("Heart_Disease_Prediction.csv")

In [3]:
df.head()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,Presence
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,Absence
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,Presence
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,Absence
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,Absence


In [4]:
df.shape

(268, 14)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268 entries, 0 to 267
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      268 non-null    int64  
 1   Sex                      268 non-null    int64  
 2   Chest pain type          268 non-null    int64  
 3   BP                       268 non-null    int64  
 4   Cholesterol              268 non-null    int64  
 5   FBS over 120             268 non-null    int64  
 6   EKG results              268 non-null    int64  
 7   Max HR                   268 non-null    int64  
 8   Exercise angina          268 non-null    int64  
 9   ST depression            268 non-null    float64
 10  Slope of ST              268 non-null    int64  
 11  Number of vessels fluro  268 non-null    int64  
 12  Thallium                 268 non-null    int64  
 13  Heart Disease            268 non-null    object 
dtypes: float64(1), int64(12), 

In [6]:
df.describe()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000,268.000000
mean,54.376866,0.675373,3.167910,131.205224,249.738806,0.149254,1.022388,149.839552,0.328358,1.050746,1.582090,0.664179,4.697761
std,9.109187,0.469111,0.950939,17.833994,51.711448,0.357005,0.997874,23.111203,0.470495,1.148472,0.615633,0.935717,1.943508
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,47.750000,0.000000,3.000000,120.000000,213.000000,0.000000,0.000000,133.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,245.000000,0.000000,2.000000,154.000000,0.000000,0.800000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,278.000000,0.000000,2.000000,166.250000,1.000000,1.650000,2.000000,1.000000,7.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000


In [7]:
df.isnull().sum()

Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [9]:
df.shape

(268, 14)

In [10]:
hh=df.select_dtypes(include=["object"]).columns
for i in hh:
    print(f"unique values in {i}")
    print(df[i].unique())
    print(".......................")

unique values in Heart Disease
['Presence' 'Absence']
.......................


In [11]:
nc=df.select_dtypes(include="number")
for i in nc:
    print(i)

Age
Sex
Chest pain type
BP
Cholesterol
FBS over 120
EKG results
Max HR
Exercise angina
ST depression
Slope of ST
Number of vessels fluro
Thallium


In [12]:
df.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease'],
      dtype='object')

In [13]:
xa=df.drop("Heart Disease",axis=1)

In [14]:
x=pd.get_dummies(data=xa,drop_first=True)
y=df["Heart Disease"]

In [15]:
trainr2=[]
testr2=[]
cv=[]

for i in range(0,100):
    from sklearn.model_selection import train_test_split
    xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=i)
    
    #modeling
    from sklearn.linear_model import LogisticRegression
    model=LogisticRegression()
    model.fit(xtrain,ytrain)
    #print(model.intercept_)
    #print(model.coef_)
    #prediction
    ypredtest=model.predict(xtest)
    ypredtrain=model.predict(xtrain)
    #evalution
    trainr2.append(model.score(xtrain,ytrain))
    testr2.append(model.score(xtest,ytest))
    from sklearn.model_selection import cross_val_score
    cv.append(cross_val_score(model,x,y,cv=5,).mean())

hh=pd.DataFrame({"train":trainr2,"test":testr2,"cv":cv})
#print(hh)
gg=hh[(abs(hh["train"]-hh["test"])<=0.05) & (abs(hh["test"]-hh["cv"])<=0.05)]
#print(gg)
a=gg[gg["test"]==gg["test"].max()].index.to_list()[0]
print(a)


6


In [16]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=6)

In [17]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
xtrain=scaler.fit_transform(xtrain)
xtest=scaler.transform(xtest)

# Logistic Regression

In [18]:
from sklearn.linear_model import LogisticRegression
lmodel=LogisticRegression()
lmodel.fit(xtrain,ytrain)


ypredtrain=lmodel.predict(xtrain)
ypredtest=lmodel.predict(xtest)

from sklearn.metrics import accuracy_score
ltr=accuracy_score(ytrain,ypredtrain)
lts=accuracy_score(ytest,ypredtest)
print("train_accuracy",accuracy_score(ytrain,ypredtrain))
print("test_accuracy",accuracy_score(ytest,ypredtest))

from sklearn.model_selection import cross_val_score
cv=cross_val_score(lmodel,x,y,cv=5)
p=cv.mean()
print("cross_val_score",p)

train_accuracy 0.8504672897196262
test_accuracy 0.8888888888888888
cross_val_score 0.8470300489168414


## Hyperparameter Tunning for KNN


In [19]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
estimater=KNeighborsClassifier()
w={"n_neighbors":list(range(1,10)),"p":[1,2]}
par=GridSearchCV(estimater,w,cv=5,scoring="accuracy")
par.fit(xtrain,ytrain)
par.best_params_

{'n_neighbors': 7, 'p': 1}

# KNN

In [20]:
from sklearn.neighbors import KNeighborsClassifier
kmodel=KNeighborsClassifier(n_neighbors=7, p=1)
kmodel.fit(xtrain,ytrain)

ypredtrain=kmodel.predict(xtrain)
ypredtest=kmodel.predict(xtest)

from sklearn.metrics import accuracy_score
a=accuracy_score(ytest,ypredtest)
c=accuracy_score(ytrain,ypredtrain)
print("train_accuracy",c)
print("test_accuracy",a)

from sklearn.model_selection import cross_val_score
cv=cross_val_score(kmodel,x,y,cv=5)
cvs=cv.mean()
print("cross_val_score",cvs)


train_accuracy 0.8364485981308412
test_accuracy 0.8703703703703703
cross_val_score 0.7199860237596087


## hyperparameter tunning for SVM

In [21]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
estimator=SVC()
parms={"C":list(range(1,10)),"kernel":["linear","rbf","sigmoid","poly"]}

ii=GridSearchCV(estimator,parms,cv=5,scoring="accuracy")
ii.fit(xtrain,ytrain)
print(ii.best_params_)

{'C': 3, 'kernel': 'sigmoid'}


In [22]:
from sklearn.svm import SVC
smodel=SVC(C=3,kernel="sigmoid")
smodel.fit(xtrain,ytrain)

ypredtrain=smodel.predict(xtrain)
ypredtest=smodel.predict(xtest)

from sklearn.metrics import accuracy_score
x2=accuracy_score(ytrain,ypredtrain)
x3=accuracy_score(ytest,ypredtest)
print("train_accuracy",x2)
print("test_accuracy",x3)

from sklearn.model_selection import cross_val_score
cv=cross_val_score(smodel,x,y,cv=5)
cvs1=cv.mean()
print("cross_val_score",cvs1)


train_accuracy 0.7897196261682243
test_accuracy 0.8333333333333334
cross_val_score 0.6606568832983928


## hyperparameter tunning for decsion tree

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
estimator=DecisionTreeClassifier()
parms={"criterion":["gini","entropy"],"max_depth":list(range(1,18))}
f=GridSearchCV(estimator,parms,scoring="accuracy",cv=5)
f.fit(xtrain,ytrain)
f.best_params_

{'criterion': 'entropy', 'max_depth': 5}

In [24]:
from sklearn.tree import DecisionTreeClassifier
dmodel=DecisionTreeClassifier(criterion="entropy",max_depth=5,random_state=0)
dmodel.fit(xtrain,ytrain)

ypredtrain=dmodel.predict(xtrain)
ypredtest=dmodel.predict(xtest)

from sklearn.metrics import accuracy_score
a1=accuracy_score(ytest,ypredtest)
c1=accuracy_score(ytrain,ypredtrain)
print("train_accuracy",c1)
print("test_accuracy",a1)

from sklearn.model_selection import cross_val_score
cv=cross_val_score(dmodel,x,y,cv=5)
cvs2=cv.mean()
print("cross_val_score",cvs2)

train_accuracy 0.9345794392523364
test_accuracy 0.7037037037037037
cross_val_score 0.7686233403214535


## hyperparameter tunning for random forest

In [25]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import GridSearchCV
estimator=RandomForestClassifier()
parms={"n_estimators":list(range(1,15))}
grid=GridSearchCV(estimator,parms,cv=5,scoring="accuracy")
grid.fit(xtrain,ytrain)
grid.best_params_

{'n_estimators': 10}

In [26]:
from sklearn.ensemble import RandomForestClassifier 
rmodel=RandomForestClassifier(n_estimators=10)
rmodel.fit(xtrain,ytrain)
ypredtrain=rmodel.predict(xtrain)
ypredtest=rmodel.predict(xtest)
from sklearn.metrics import accuracy_score
tra=accuracy_score(ytrain,ypredtrain)
tsa=accuracy_score(ytest,ypredtest)
print("train_accuracy",tra)
print("test_accuracy",tsa)
from sklearn.model_selection import cross_val_score
cv=cross_val_score(rmodel,x,y,cv=5)
p1=cv.mean()
print("cross_val_score",p1)


train_accuracy 0.9906542056074766
test_accuracy 0.7407407407407407
cross_val_score 0.7874912648497554


## hyperparameter tunning for ada boost

In [27]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
estimator=AdaBoostClassifier()
parms={"n_estimators":list(range(1,10)),"learning_rate":[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]}
grid=GridSearchCV(estimator,parms,cv=5,scoring="accuracy")
grid.fit(xtrain,ytrain)
grid.best_params_

{'learning_rate': 0.8, 'n_estimators': 4}

In [29]:
from sklearn.ensemble import AdaBoostClassifier
amodel=AdaBoostClassifier(n_estimators=4,learning_rate=0.8)
amodel.fit(xtrain,ytrain)
ypredtrain=amodel.predict(xtrain)
ypredtest=amodel.predict(xtest)
from sklearn.metrics import accuracy_score
tra1=accuracy_score(ytrain,ypredtrain)
tsa1=accuracy_score(ytest,ypredtest)
print("train_accuracy",tra1)
print("test_accuracy",tsa1)
from sklearn.model_selection import cross_val_score
cv=cross_val_score(amodel,x,y,cv=5)
p2=cv.mean()
print("cross_val_score",p2)


train_accuracy 0.8504672897196262
test_accuracy 0.9074074074074074
cross_val_score 0.8323549965059399


## hyperparameter tunning for gradient boost

In [30]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
estimator=GradientBoostingClassifier()
parms={"n_estimators":[2,14,12,20,30],"learning_rate":[0.1,0.2,0.6,0.8,1]}
grid=GridSearchCV(estimator,parms,cv=5,scoring="accuracy")
grid.fit(xtrain, ytrain)

grid.best_params_

{'learning_rate': 0.1, 'n_estimators': 20}

In [31]:
from sklearn.ensemble import GradientBoostingClassifier
gmodel=GradientBoostingClassifier(n_estimators=20,learning_rate=0.1)
gmodel.fit(xtrain,ytrain)
ypredtrain=gmodel.predict(xtrain)
ypredtest=gmodel.predict(xtest)
from sklearn.metrics import accuracy_score
tra2=accuracy_score(ytrain,ypredtrain)
tsa2=accuracy_score(ytest,ypredtest)
print("gdtrain_accuracy",tra2)
print("gdtest_accuracy",tsa2)
from sklearn.model_selection import cross_val_score
cv=cross_val_score(gmodel,x,y,cv=5)
p3=cv.mean()
print("gdcross_val_score",p3)


gdtrain_accuracy 0.9252336448598131
gdtest_accuracy 0.7777777777777778
gdcross_val_score 0.7986722571628233


In [32]:
x1=pd.get_dummies(data=xa,drop_first=True)
df["Heart Disease"].replace({"Absence":0,"Presence":1},inplace=True)
y1=df["Heart Disease"]


In [33]:
from sklearn.model_selection import train_test_split
x1train,x1test,y1train,y1test=train_test_split(x1,y1,test_size=0.2,random_state=6)

In [34]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x1train=scaler.fit_transform(x1train)
x1test=scaler.transform(x1test)

## hyper paameter tunning for xgboost

In [35]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
estimator=XGBClassifier()
parms={"n_estimators":[2,14,12,20,30],"max_depth":[1,4,7,10,15],"gamma":[0,0.1,0.2,0.3,0.5,1]}
grid=GridSearchCV(estimator,parms,cv=5,scoring="accuracy")
grid.fit(x1train, y1train)

grid.best_params_

{'gamma': 0, 'max_depth': 1, 'n_estimators': 14}

In [36]:
from xgboost import XGBClassifier
xmodel=XGBClassifier(gamma=0,max_depth=1,n_estimators=14)
xmodel.fit(x1train,y1train)
ypredtrain=xmodel.predict(x1train)
ypredtest=xmodel.predict(x1test)
from sklearn.metrics import accuracy_score
tra3=accuracy_score(y1train,ypredtrain)
tsa3=accuracy_score(y1test,ypredtest)
print("train_accuracy",tra3)
print("test_accuracy",tsa3)
from sklearn.model_selection import cross_val_score
cv=cross_val_score(xmodel,x1,y1,cv=5)
p4=cv.mean()
print("cross_val_score",p4)


train_accuracy 0.8691588785046729
test_accuracy 0.8333333333333334
cross_val_score 0.8433263452131378


In [37]:
final=pd.DataFrame({"Model":["LogisticRegression","KNN","SVM","DescionTree","RandomForest","Adaboost","Gradient","xg"],
                    "train_accuracy":[ltr,c,x2,c1,tra,tra1,tra2,tra3],
                   "test_accuracy":[lts,a,x3,a1,tsa,tsa1,tsa2,tsa3],
                   "cross_validation_score":[p,cvs,cvs1,cvs2,p1,p2,p3,p4]})

In [38]:
final

,Model,train_accuracy,test_accuracy,cross_validation_score
0,LogisticRegression,0.850467,0.888889,0.847030
1,KNN,0.836449,0.870370,0.719986
2,SVM,0.789720,0.833333,0.660657
3,DescionTree,0.934579,0.703704,0.768623
4,RandomForest,0.990654,0.740741,0.787491
5,Adaboost,0.850467,0.907407,0.832355
6,Gradient,0.925234,0.777778,0.798672
7,xg,0.869159,0.833333,0.843326


# By this we can conclude that Logistic regression is best model

In [39]:
x=pd.get_dummies(data=xa,drop_first=True)
y=df["Heart Disease"]

In [40]:
y = pd.get_dummies(y, drop_first=True)

In [41]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [42]:
model = Sequential()
model.add(Dense(64, input_dim=13, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))


In [43]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [44]:
# Train the model
model.fit(x, y, epochs=100, batch_size=32, validation_split=0.2)

Epoch 1/100


7/7 [==============================] - 3s 85ms/step - loss: 2.8760 - accuracy: 0.5421 - val_loss: 0.7910 - val_accuracy: 0.6481
Epoch 2/100
7/7 [==============================] - 0s 17ms/step - loss: 1.1693 - accuracy: 0.6402 - val_loss: 0.6836 - val_accuracy: 0.7037
Epoch 3/100
7/7 [==============================] - 0s 18ms/step - loss: 1.0647 - accuracy: 0.5888 - val_loss: 0.5930 - val_accuracy: 0.7407
Epoch 4/100
7/7 [==============================] - 0s 18ms/step - loss: 0.9491 - accuracy: 0.6542 - val_loss: 0.7047 - val_accuracy: 0.6111
Epoch 5/100
7/7 [==============================] - 0s 18ms/step - loss: 0.8167 - accuracy: 0.6402 - val_loss: 0.7512 - val_accuracy: 0.7037
Epoch 6/100
7/7 [==============================] - 0s 17ms/step - loss: 0.7524 - accuracy: 0.6542 - val_loss: 0.6006 - val_accuracy: 0.6111
Epoch 7/100
7/7 [==============================] - 0s 18ms/step - loss: 0.7237 - accuracy: 0.6916 - val_loss: 0.5882 - val_accuracy: 0.5741
Epoch 8/100
7/7 [=

Epoch 56/100
7/7 [==============================] - 0s 22ms/step - loss: 0.4122 - accuracy: 0.8037 - val_loss: 0.4971 - val_accuracy: 0.7778
Epoch 57/100
7/7 [==============================] - 0s 19ms/step - loss: 0.4385 - accuracy: 0.7897 - val_loss: 0.6605 - val_accuracy: 0.7222
Epoch 58/100
7/7 [==============================] - 0s 19ms/step - loss: 0.4735 - accuracy: 0.7897 - val_loss: 0.4499 - val_accuracy: 0.7407
Epoch 59/100
7/7 [==============================] - 0s 22ms/step - loss: 0.4139 - accuracy: 0.8271 - val_loss: 0.4501 - val_accuracy: 0.7593
Epoch 60/100
7/7 [==============================] - 0s 18ms/step - loss: 0.3557 - accuracy: 0.8505 - val_loss: 0.4585 - val_accuracy: 0.8148
Epoch 61/100
7/7 [==============================] - 0s 17ms/step - loss: 0.3481 - accuracy: 0.8598 - val_loss: 0.4152 - val_accuracy: 0.8148
Epoch 62/100
7/7 [==============================] - 0s 17ms/step - loss: 0.3805 - accuracy: 0.8224 - val_loss: 0.4329 - val_accuracy: 0.7593
Epoch 63/100


In [45]:
loss, accuracy = model.evaluate(x, y)
print(f'Model Accuracy: {accuracy * 100:.2f}%')

9/9 [==============================] - 0s 5ms/step - loss: 0.3770 - accuracy: 0.8433
Model Accuracy: 84.33%
